In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils import (
    build_binary_relevance_maps,
    build_eval_items,
    build_global_corpus,
    evaluate_sparqlce_checkpoint,
    load_triplets,
)

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = os.path.join(PROJECT_ROOT, "dataset", "hard_triplets_tfidf_dbpedia.json")
CHECKPOINT_PATH = os.path.join(PROJECT_ROOT, "../checkpoints", "dbpedia_model_semantic-20.pt")
TOPKS = [3, 5, 10, 20]
BATCH = 32

raw_data = load_triplets(DATA_PATH)
global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Loaded {len(raw_data)} items")
print(f"Global corpus size: {len(global_corpus)}")
print(f"Eval anchors with non-empty positives: {len(eval_items)}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")

Loaded 337 items
Global corpus size: 1677
Eval anchors with non-empty positives: 337
Checkpoint path: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/evaluation/../checkpoints/dbpedia_model_semantic-20.pt


In [3]:
evaluate_sparqlce_checkpoint(
    checkpoint_path=CHECKPOINT_PATH,
    eval_items=eval_items,
    global_corpus=global_corpus,
    relevance_scores_per_item=relevance_scores_per_item,
    topks=TOPKS,
    batch_size=BATCH,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35879.57it/s]


Loading model from /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/evaluation/../checkpoints/dbpedia_model_semantic-20.pt...


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
        pool_request.request
    )
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/httpcore/_sync/connection.py", line 103, in


SPARQLCE
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1068  0.0404  0.0528  0.1424  0.2077    0.1068  0.1037
@5     0.1134  0.0689  0.0769  0.1574  0.2730    0.1134  0.1151
@10    0.1160  0.1319  0.1115  0.1739  0.3976    0.1160  0.1339
@20    0.1083  0.2325  0.1350  0.1841  0.5430    0.1083  0.1731
